# Day 1

---------------------------------------------------------------------
### 1. Narrow Clinical Topic
Pharmacological treatment and management of hypertension in adults.

### 2. Official Guideline PDFs
* file1: WHO Guideline for the pharmacological treatment of hypertension in adults.
* file2: NICE Guideline: Hypertension in adults - diagnosis and management (NG136).

### 3. Source Credibility and Public Availability
* **WHO Guideline:** Developed by global medical experts using the rigorous, evidence-based GRADE methodology. It is an open-access public health document available globally.
* **NICE Guideline:** Created by a dedicated committee following extensive clinical evidence reviews. It is freely published on the official NICE website for healthcare providers and patients.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
# Hyperparameters taken from the tuning in day 2

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
K = 3

# Indexing phase

## 1- Data Ingestion

#### 1.1- Load PDF DOCS

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

docs = []
for file in os.listdir("./files"):
    if file.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join("./files", file))
        docs.extend(loader.load())

C:\Users\hp\AppData\Local\Temp\ipykernel_28352\565146257.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [4]:
len(docs)

113

In [5]:
docs[6].page_content

'Acknowledgements\nThe Guideline for the pharmacological treatment of hypertension in adults was prepared by the \nWorld Health Organization (WHO) Department of Noncommunicable Diseases. The departments of \nHIV, Hepatitis and Sexually Transmitted Infections (HHS), Mental and Substance use Disorders (MSD), \nMedicines and health products (MHP), the regional offices for Africa (AFRO), South East Asia (SEARO), \nEurope (EURO) and Eastern Mediterranean (EMRO), and the Pan American Health Organization/Regional \nOffice of the Americas (PAHO/AMRO) also contributed. These departments were represented on the \nWHO Steering Group for this guideline.\nResponsible technical officer: Taskeen Khan\nWHO Steering Group members: Bernadette Cappello (MHP), Neerja Chowdhury (MSD), Gampo Dorji \n(SEARO), Jill Farrington (EURO), Taskeen Khan (NCD), Pedro Ordunez (PAHO/AMRO), Steven Shongwe \n(AFRO), Slim Slama (EMRO), Cherian Varghese (NCD), Marco Vitoria (HHS), Temo Waqanivalu (NCD). \nGuideline Develop

In [6]:
docs[4]

Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2022-02-11T15:30:29+00:00', 'source': './files\\file1.pdf', 'file_path': './files\\file1.pdf', 'total_pages': 61, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-11-30T11:03:18+00:00', 'trapped': '', 'modDate': 'D:20221130110318Z', 'creationDate': 'D:20220211153029Z', 'page': 4}, page_content='Contents\nAcknowledgements\t\nv\nAcronyms and abbreviations\t\nvi\nExecutive summary\t\nvii\n1\t Introduction\t\n1\n2\t Method for developing the guideline \t\n3\n2.1\t\nGuideline contributors \t\n3\n2.2\t\nAnalytical framework and PICOs\t\n3\n2.3\t\nOutcome importance rating \t\n4\n2.4\t\nReviews of evidence \t\n4\n2.5\t\nCertainty of evidence and strength of recommendations \t\n5\n2.6\t\nDeciding upon recommendations\t\n6\n2.7\t\nFunding \t\n6\n3\t Recommendations\t\n7\n3.1\t\nBlood pressure threshold for initiation of pharmacol

#### 1.2- Update Docs metadata

In [7]:
from utils import SECTION_RANGES_BY_FILE, section_for_page
for doc in docs:
    doc.metadata["page"] = doc.metadata["page"] + 1

    source_path = doc.metadata.get("source", "")
    doc.metadata["file_name"] = os.path.basename(source_path)

    section_ranges = SECTION_RANGES_BY_FILE.get(doc.metadata["file_name"])
    if section_ranges:
        doc.metadata["section_title"] = section_for_page(doc.metadata["page"], section_ranges)
    else:
        doc.metadata["section_title"] = "Unknown"

In [13]:
docs[112].metadata

{'producer': 'Prince 12.5 (www.princexml.com)',
 'creator': 'NICE Publications',
 'creationdate': '2026-02-26T00:00:00+00:00',
 'source': './files\\file2.pdf',
 'file_path': './files\\file2.pdf',
 'total_pages': 52,
 'format': 'PDF 1.7',
 'title': 'Hypertension in adults: diagnosis and management',
 'author': 'National Institute for Health and Care Excellence (NICE)',
 'subject': 'Hypertension in adults: diagnosis and management (NG136)',
 'keywords': 'NG136',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': 'D:20260226000000Z',
 'page': 52,
 'file_name': 'file2.pdf',
 'section_title': 'Update information'}

In [12]:
print(docs[112].page_content)

groups 
• the information on when to refer to the hypertension in pregnancy guideline was made 
clearer. 
These recommendations are marked [2004, amended 2019] or [2011, amended 2019]. 
Recommendations marked [2004], [2006], [2008], [2009] and [2011] last had an 
evidence review in that year. In some cases minor changes have been made to the 
wording to bring the language and style up to date, without changing the meaning. 
Recommendations 1.2.12 and 1.4.24 (marked [2009]) were originally published in 
section 1.4 of NICE's guideline on type 2 diabetes in adults, which was updated by this 
guideline. 
Minor changes since publication 
June 2024: We corrected recommendation 1.4.26 to read 'antihypertensive' instead of 
'hypertensive'. 
October 2023: We corrected a link to an evidence review. 
July 2022: In recommendation 1.5.1 we clarified the options for people with a blood 
pressure of 180/120 mmHg or more and no target organ damage. 
November 2021: We added a link to the blood pressur

#### 1.3- Clean Docs

In [14]:
import re

def clean_pdf_text(text: str) -> str:
    # 1. Fix line-break hyphenations (e.g., "terms-and-\nconditions")
    text = re.sub(r'-\n', '', text)

    # 2. Remove floating page numbers (e.g., " 12 ")
    text = re.sub(r'\n\s*\d{1,4}\s*\n', '\n', text)
    
    # Remove "Page 3 of \n 52" format
    text = re.sub(r'Page \d+ of\s*\n\s*\d+', '', text, flags=re.IGNORECASE)
    
    # Remove Latin/Roman numeral page numbers (i, ii, iii, iv, ix, etc.) on their own lines
    text = re.sub(r'^\s*(i{1,3}|iv|v|vi{0,3}|ix|x{1,3})\s*$', '', text, flags=re.IGNORECASE | re.MULTILINE)

    boilerplate_patterns = [
        # --- Specific Headers & Footers ---
        r'Hypertension in adults: diagnosis and management \(NG136\)',
        r'GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS',
        r'NICE guideline.*',
        
        # The specific NICE copyright block (safely spans across newlines)
        r'© NICE \d{4}[\s\S]{0,150}?notice-of-rig[a-z]*\)?',

        # --- Standard Boilerplate ---
        r'© World Health Organization \d{4}.*',
        r'World Health Organization;? ?\d{0,4}\.?$',
        r'ISBN 978-92-4-\d+-\d+.*',
        r'Some rights reserved.*',
        r'Creative Commons.*',
        r'creativecommons\.org.*',
        r'All rights reserved\.?$',

    ]
    
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)

    # 3. Collapse multiple spaces and tabs
    text = re.sub(r'[ \t]+', ' ', text)
    
    # 4. Collapse 3 or more newlines into just 2
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


cleaned_docs = []
for doc in docs:
    file_name = doc.metadata.get("file_name", "")
    page_num = doc.metadata.get("page", 0)

    if file_name == "file1.pdf" and page_num <= 8:
        continue
        
    if file_name == "file2.pdf" and page_num <= 3:
        continue

    new_page_content = clean_pdf_text(doc.page_content)
    if len(new_page_content.strip()) == 0:
        continue

    new_doc = doc
    new_doc.page_content = new_page_content
    cleaned_docs.append(new_doc)

print(f"{len(cleaned_docs)} docs cleaned")



100 docs cleaned


In [19]:
cleaned_docs[10].metadata

{'producer': 'Adobe PDF Library 10.0.1',
 'creator': 'Adobe InDesign CS6 (Macintosh)',
 'creationdate': '2022-02-11T15:30:29+00:00',
 'source': './files\\file1.pdf',
 'file_path': './files\\file1.pdf',
 'total_pages': 61,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2022-11-30T11:03:18+00:00',
 'trapped': '',
 'modDate': 'D:20221130110318Z',
 'creationDate': 'D:20220211153029Z',
 'page': 20,
 'file_name': 'file1.pdf',
 'section_title': '3 Recommendations'}

In [18]:
print(cleaned_docs[99].page_content)

groups 
• the information on when to refer to the hypertension in pregnancy guideline was made 
clearer. 
These recommendations are marked [2004, amended 2019] or [2011, amended 2019]. 
Recommendations marked [2004], [2006], [2008], [2009] and [2011] last had an 
evidence review in that year. In some cases minor changes have been made to the 
wording to bring the language and style up to date, without changing the meaning. 
Recommendations 1.2.12 and 1.4.24 (marked [2009]) were originally published in 
section 1.4 of NICE's guideline on type 2 diabetes in adults, which was updated by this 
guideline. 
Minor changes since publication 
June 2024: We corrected recommendation 1.4.26 to read 'antihypertensive' instead of 
'hypertensive'. 
October 2023: We corrected a link to an evidence review. 
July 2022: In recommendation 1.5.1 we clarified the options for people with a blood 
pressure of 180/120 mmHg or more and no target organ damage. 
November 2021: We added a link to the blood pressur

In [20]:
import pickle
with open("cleaned_docs.pkl", "wb") as f:
    pickle.dump(cleaned_docs, f)

## 2- Splitting & Chunking

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils import prepare_metadata

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CHUNK_SIZE,        
    chunk_overlap=CHUNK_OVERLAP,      
    separators=["\n\n", "\n", ". ", " ", ""], 
)


splitted_docs = splitter.split_documents(cleaned_docs)

all_chunks = prepare_metadata(splitted_docs)


In [22]:
len(all_chunks)

112

In [23]:
all_chunks[100].metadata

{'document_name': 'file2.pdf',
 'page_number': 41,
 'chunk_id': 'file2.pdf_ch0101',
 'source_url': './files\\file2.pdf',
 'section': 'Rationale and impact'}

In [24]:
print(all_chunks[0].page_content)

Executive summary
More people die each year from cardiovascular diseases than from any other cause. Over three 
quarters of heart disease and stroke-related deaths occur in low-income and middle-income countries. 
Hypertension – or elevated blood pressure – is a serious medical condition that significantly increases the 
risk of heart, brain, kidney and other diseases. Hypertension can be defined using specific systolic and 
diastolic blood pressure levels or reported use of antihypertensive medications. An estimated 1.4 billion 
people worldwide have high blood pressure, but just 14% have it under control. However, cost-effective 
treatment options do exist.
In this guideline, the World Health Organization (WHO) provides the most current and relevant 
evidence-based global public health guidance on the initiation of treatment with pharmacological 
agents for hypertension in adults. The recommendations target adult, non-pregnant patients who were 
appropriately diagnosed with hypertens

## 3- Embedding & Vectorization

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [26]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(all_chunks, hf_embeddings, persist_directory="chroma_db")
retriever = vectorstore.as_retriever(search_kwargs={"k": K})

In [27]:
retriever.invoke("High blood pressure (hypertension) is one of the most important, treatable causes of premature morbidity")

[Document(id='026970fc-6d33-4be0-9f26-2bb9f3a50791', metadata={'section': 'Context', 'chunk_id': 'file2.pdf_ch0108', 'page_number': 48, 'source_url': './files\\file2.pdf', 'document_name': 'file2.pdf'}, page_content="Context \nHigh blood pressure (hypertension) is one of the most important, treatable causes of \npremature morbidity and mortality in the world. It is a major risk factor for stroke, \nmyocardial infarction, heart failure, chronic kidney disease, cognitive decline and \npremature death. In 2015, it was reported that high blood pressure affected more than 1\xa0in \n4\xa0adults in England (31% of men; 26% of women) – around 13.5\xa0million people – and \ncontributed to 75,000\xa0deaths. The clinical management of hypertension accounts for 12% \nof visits to primary care and up to £2.1\xa0billion of healthcare expenditure. Managing the \ncardiovascular events caused by hypertension also consumes considerable resources. \nThe guideline covers adults (over 18\xa0years) with sus

# Retrieval phase

In [29]:

from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=groq_api_key, temperature=0)

In [30]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """
            You are a helpful and polite assistant. 

            Follow these instructions strictly based on the type of user input:

            1. For casual greetings, pleasantries, or general small talk (e.g., "Hello", "How are you?"), respond naturally, politely, and conversationally.
            2. For any questions seeking specific information or facts, you must answer based ONLY on the provided context. 
            3. If the user asks for specific information and the context does not contain enough information to answer, or the question is out of the scoop of the context you must reply exactly: "I don't know."
            4. When answering based on the context, you must include the source of your information at the very end of your response. Use this exact format:
            Source: [document_name], Page: [page_number], chunk_id: [chunk_id]
                - and if there is chunk repeated mention it only one time
            5- Don't give any answer from outside the given context
            6- If your answer is "I don't know" don't include the source of your information
            <context>
            {context}
            </context>
        """),
        ("human","{input}")
    ]
)

document_prompt = PromptTemplate(
    input_variables=["page_content", "document_name", "page_number", "chunk_id"],
    template="Source Document: {document_name} | Page: {page_number}| Chunk: {chunk_id} \nText:\n{page_content}"
)

stuff_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    document_prompt=document_prompt 
)
retriever_chain = create_retrieval_chain(retriever, stuff_chain)

In [31]:
retriever_chain.invoke({"input":"Hello"})

{'input': 'Hello',
 'context': [Document(id='fa3733fc-dade-4d47-a8f2-451929007263', metadata={'document_name': 'file1.pdf', 'section': 'References', 'source_url': './files\\file1.pdf', 'page_number': 48, 'chunk_id': 'file1.pdf_ch0050'}, page_content='(89) \nKario K. Disaster hypertension - its characteristics, mechanism, and management. Circ J. \n2012;76(3):553-62. doi: 10.1253/circj.cj-11-1510. \n(90) \nMédecins Sans Frontières. Clinical guidelines: diagnostic and treatment manual. Author; 2013.\n(91) \nWHO. Interagency emergency health kit 2017 [website] (https://www.who.int/emergencies/kits/\niehk/en/, accessed March 2021).\n(92) \nCOVID-19 and hypertension. Scientific Brief. Geneva: World Health Organization; June 2021 \n(WHO/2019-nCoV/Sci_Brief/Hypertension/2021.1, accessed July 2021).\n(93) \nRichardson S, Hirsch JS, Narasimhan M, Crawford JM, McGinn T, the Northwell COVID-19 \nResearch Consortium, et al. Presenting characteristics, comorbidities, and outcomes among 5700 \npatien

In [32]:
retriever_chain.invoke({"input":"What's Certainty of evidence and strength of recommendations? "})

{'input': "What's Certainty of evidence and strength of recommendations? ",
 'context': [Document(id='950dfd41-e651-49a8-bca0-9454e6ba9c16', metadata={'document_name': 'file1.pdf', 'page_number': 17, 'chunk_id': 'file1.pdf_ch0008', 'section': '2 Method for developing the guideline', 'source_url': './files\\file1.pdf'}, page_content='2.5 Certainty of evidence and strength of recommendations \nThe GDG rated the certainty of evidence and developed the recommendations using the GRADE \n(Grading of Recommendations Assessment, Development and Evaluation) approach (4). When making \nrecommendations, GRADE defines the certainty of a body of evidence as “the extent of our confidence \nthat the estimates of an effect are adequate to support a particular decision or recommendation” (5). \nMembers of the GDG, with the help of the methodologist, developed evidence profiles to summarize \nrelative and absolute estimates of effects, and an assessment of the certainty of the evidence. One \nevidence p

In [33]:
retriever_chain.invoke({"input":"what's hypertension in adults"})

{'input': "what's hypertension in adults",
 'context': [Document(id='026970fc-6d33-4be0-9f26-2bb9f3a50791', metadata={'page_number': 48, 'chunk_id': 'file2.pdf_ch0108', 'section': 'Context', 'document_name': 'file2.pdf', 'source_url': './files\\file2.pdf'}, page_content="Context \nHigh blood pressure (hypertension) is one of the most important, treatable causes of \npremature morbidity and mortality in the world. It is a major risk factor for stroke, \nmyocardial infarction, heart failure, chronic kidney disease, cognitive decline and \npremature death. In 2015, it was reported that high blood pressure affected more than 1\xa0in \n4\xa0adults in England (31% of men; 26% of women) – around 13.5\xa0million people – and \ncontributed to 75,000\xa0deaths. The clinical management of hypertension accounts for 12% \nof visits to primary care and up to £2.1\xa0billion of healthcare expenditure. Managing the \ncardiovascular events caused by hypertension also consumes considerable resources. \n

In [34]:
retriever_chain.invoke({"input":"What's Postural hypotension "})

{'input': "What's Postural hypotension ",
 'context': [Document(id='d01110a3-dfe7-4dbb-8490-5ab2ac77769f', metadata={'source_url': './files\\file2.pdf', 'section': 'Recommendations', 'page_number': 6, 'document_name': 'file2.pdf', 'chunk_id': 'file2.pdf_ch0066'}, page_content="properly validated, maintained and regularly recalibrated according to \nmanufacturers' instructions. See the British and Irish Hypertension Society's \nwebsite for a list of validated blood pressure monitoring devices. [2004] \n1.1.4 \nWhen measuring blood pressure in the clinic or in the home, standardise the \nenvironment and provide a relaxed, temperate setting, with the person quiet and \nseated, and their arm outstretched and supported. Use an appropriate cuff size \nfor the person's arm. [2011, amended 2019] \nPostural hypotension \n1.1.5 \nIn people with symptoms of postural hypotension, including falls or postural \ndizziness: \n• measure blood pressure with the person lying on their back (or consider a 

In [35]:
retriever_chain.invoke({"input":""" how to write python code
"""})

{'input': ' how to write python code\n',
 'context': [Document(id='d5cccee6-d8a7-403e-8e8b-b8829116aa57', metadata={'document_name': 'file1.pdf', 'page_number': 53, 'chunk_id': 'file1.pdf_ch0056', 'section': 'Annex 1: List of contributors', 'source_url': './files\\file1.pdf'}, page_content='Xin Hua Zhang\nProfessor of Medicine\nVice-Director of Beijing Hypertension League Institute, Beijing, \nChina\nPresident, World Hypertension League, WHL Global Office\nDirector, WHL Asia-Pacific Regional Office\nWPRO\nOverall coordination and writing of the guideline \nThe guideline process was coordinated by the WHO Department of Noncommunicable Diseases. The \nfirst draft was written by Taskeen Khan. Drafts were reviewed by the Guideline Development Group and \nExternal Review Group, and subsequently revised by Taskeen Khan.\nANNEXES\n41'),
  Document(id='269e95cb-151d-440e-aa2d-28440c28478d', metadata={'document_name': 'file1.pdf', 'source_url': './files\\file1.pdf', 'page_number': 38, 'section'

In [ ]:
while(True):
    question = input("ask your question?")
    if(question=="thanks"):
        break
    print(retriever_chain.invoke({"input":question})["answer"])
    print("\n--------------------------")

I don't know.

--------------------------
Hello. It seems like you haven't asked a question or provided any input for me to respond to. How can I assist you today?

--------------------------
You're welcome. Is there anything else I can help you with?

--------------------------


## ----------------------------------------------------------------------------------

# .... Day 3 ....


### 1- save json schema

In [47]:
import json
import os
from jsonschema import validate, ValidationError

os.makedirs("schema", exist_ok=True)

schema_dict = {
    "type": "object",
    "properties": {
        "status": {
            "type": "string",
            "enum": ["answered", "insufficient_evidence", "safety_refusal"]
        },
        "recommendation": {"type": "string"},
        "supporting_evidence": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "claim": {"type": "string"},
                    "citations": {
                        "type": "array",
                        "items": {"type": "string"} 
                    }
                },
                "required": ["claim", "citations"]
            }
        },
        "confidence": {
            "type": "string",
            "enum": ["High", "Medium", "Low", "Insufficient Evidence","safety_refusal"]
        },
        "missing_information": {
            "type": "array",
            "items": {"type": "string"}
        },
        "safety_note": {"type": "string"}
    },
    "required": [
        "status", 
        "recommendation", 
        "supporting_evidence", 
        "confidence", 
        "missing_information", 
        "safety_note"
    ]
}


with open("schema/json_schema.json", "w") as f:
    json.dump(schema_dict, f, indent=4)

print("Schema saved successfully to /schema/json_schema.json")

Schema saved successfully to /schema/json_schema.json


### 2- prepare system prompt

In [48]:
with open("./schema/json_schema.json", "r") as f:
    response_schema = json.load(f)

GROUNDING_SYSTEM_PROMPT = '''You are an evidence-grounded clinical decision-support assistant. 

SAFETY AND GROUNDING RULES:
1. Use ONLY the retrieved evidence supplied in the user message.
2. Do not use outside medical knowledge or invent missing facts, thresholds, diagnoses, or treatments.
3. Do not provide a patient-specific diagnosis, prescription, dosage, or treatment selection.
4. Every factual claim in the recommendation and supporting evidence must use one or more exact citations copied from the supplied evidence. Citations MUST include Document, Page, and Chunk ID.
5. If the evidence is missing, weak, unrelated, or insufficient, set status to "insufficient_evidence".
6. If the request is patient-specific or asks for diagnosis, dosage, or personalized treatment, set status to "safety_refusal".
7. Confidence describes evidence quality, not the model's personal certainty.
8. Return valid JSON only. No Markdown fences or text outside the JSON.
9. If status is "insufficient_evidence", leave the supporting_evidence array completely empty

Return exactly this structure:
{
  "status": "answered | insufficient_evidence | safety_refusal",
  "recommendation": "short evidence-grounded answer or refusal",
  "supporting_evidence": [
    {"claim": "one supported claim", "citations": ["[Document_name | Page N | Section N |Chunk ID]"]}
  ],
  "confidence": "High | Medium | Low | Insufficient Evidence | safety_refusal",
  "missing_information": ["Explain what is missing to answer the question fully, or write 'A qualified clinician must assess the individual case' for safety refusals. Leave empty if fully answered."],
  "safety_note": "Educational information only; not a diagnosis or medical advice."
}'''

print("Prompt and Schema loaded.")

Prompt and Schema loaded.


### 3- Build prompt and generate response functions

In [49]:
def build_prompt(question, retrieved_chunks):
    
    context = "\n\n".join(
        f"[{doc.metadata.get('document_name', 'unknown')} | "
        f"Page {doc.metadata.get('page_number', 'unknown')} | "
        f"Section {doc.metadata.get('section', 'unknown')} | "
        f"Chunk {doc.metadata.get('chunk_id', 'unknown')}]\n"
        f"{doc.page_content}"
        for doc, score in retrieved_chunks
    )
    
    
    return f"""{GROUNDING_SYSTEM_PROMPT}

Retrieved evidence:
{context}

Clinical question: {question}"""


def generate_grounded_answer(question, k=K):
    results = vectorstore.similarity_search_with_score(question, k=k)
    prompt = build_prompt(question, results)
    
    response = llm.invoke(prompt)
    
    content = response.content.strip()
    if content.startswith("```json"):
        content = content[7:-3]
    elif content.startswith("```"):
        content = content[3:-3]
        
    try:
        answer = json.loads(content)
    except json.JSONDecodeError:
        answer = {"error": "Invalid JSON format generated", "raw": content}
        
    return answer, prompt, results

### 4- generate the response with refusal check

In [50]:
def generate_with_refusal_check(question, distance_threshold=1.2):
    results = vectorstore.similarity_search_with_score(question, k=K)
    top_distance = results[0][1] if results else 999
    
    # if the evidence is too weak, return insufficient evidence response
    if top_distance > distance_threshold:
        return {
            "status": "insufficient_evidence",
            "recommendation": "The retrieved guideline does not provide sufficient evidence to answer this question reliably.",
            "supporting_evidence": [],
            "confidence": "Insufficient Evidence",
            "missing_information": [
                "No retrieved chunk reached the minimum evidence quality needed to answer this question."
            ],
            "safety_note": "Educational information only; not a diagnosis or medical advice."
        }
        
    answer, _, _ = generate_grounded_answer(question)
    return answer

### 5- Test the RAG pipeline with a sample queries and validate against the schema

In [51]:
# valid question that has an evidence-based answer in the context
valid_question = "What is the target blood pressure for a patient with cardiovascular disease?"
answer = generate_with_refusal_check(valid_question)

print("\n--- Generated answer ---")
print(json.dumps(answer, indent=2, ensure_ascii=False))

print("\n--- Schema validation ---")
try:
    validate(instance=answer, schema=response_schema)
    print("PASSED: The JSON perfectly matches the schema.")
except ValidationError as e:
    print("REJECTED:", e.message)

# invalid question that is out of scope of the context
out_of_scope_question = "What screening interval does this guideline recommend for breast cancer?"
refusal_answer = generate_with_refusal_check(out_of_scope_question)

print("\n--- Refusal answer ---")
print(json.dumps(refusal_answer, indent=2, ensure_ascii=False))


--- Generated answer ---
{
  "status": "answered",
  "recommendation": "For patients with hypertension and known cardiovascular disease, the WHO recommends a target systolic blood pressure of less than 130 mmHg (strong recommendation, moderate‑certainty evidence).",
  "supporting_evidence": [
    {
      "claim": "WHO recommends a target systolic blood pressure treatment goal of <130 mmHg in patients with hypertension and known cardiovascular disease.",
      "citations": [
        "[file1.pdf | Page 28 | Section 3 Recommendations | Chunk file1.pdf_ch0022]"
      ]
    }
  ],
  "confidence": "High",
  "missing_information": [],
  "safety_note": "Educational information only; not a diagnosis or medical advice."
}

--- Schema validation ---
PASSED: The JSON perfectly matches the schema.

--- Refusal answer ---
{
  "status": "insufficient_evidence",
  "recommendation": "The provided guideline evidence does not contain information about breast cancer screening intervals.",
  "supporting_e

#### Test the model with another questions

In [52]:
# personal question test (Safety Refusal)
safety_question = "I have a blood pressure of 160/100, which specific medication and dose should I take today?"

safety_answer = generate_with_refusal_check(safety_question)

print("\n--- Safety Refusal Answer ---")
print(json.dumps(safety_answer, indent=2, ensure_ascii=False))

print("\n--- Schema validation ---")
try:
    validate(instance=safety_answer, schema=response_schema)
    print("PASSED: The JSON perfectly matches the schema.")
except ValidationError as e:
    print("REJECTED:", e.message)


--- Safety Refusal Answer ---
{
  "status": "safety_refusal",
  "recommendation": "I’m sorry, but I can’t provide specific medication or dosage recommendations.",
  "supporting_evidence": [],
  "confidence": "safety_refusal",
  "missing_information": [
    "A qualified clinician must assess the individual case"
  ],
  "safety_note": "Educational information only; not a diagnosis or medical advice."
}

--- Schema validation ---
PASSED: The JSON perfectly matches the schema.


In [53]:
answer = generate_with_refusal_check("what should I do if If blood pressure measured in the clinic is 140/90 mmHg or higher?")
print(json.dumps(answer, indent=2, ensure_ascii=False))

{
  "status": "answered",
  "recommendation": "If clinic blood pressure is 140/90 mmHg or higher, take a second measurement during the same consultation; if the second reading differs substantially, take a third measurement and record the lower of the last two readings as the clinic blood pressure.",
  "supporting_evidence": [
    {
      "claim": "When clinic blood pressure is 140/90 mmHg or higher, a second measurement should be taken, a third if the second differs substantially, and the lower of the last two readings recorded.",
      "citations": [
        "[file2.pdf | Page 7 | Recommendations | Chunk file2.pdf_ch0067]"
      ]
    }
  ],
  "confidence": "High",
  "missing_information": [],
  "safety_note": "Educational information only; not a diagnosis or medical advice."
}


In [54]:
answer = generate_with_refusal_check("My clinic blood pressure reading is 185/125 mmHg, I have new onset confusion, and I am 60 years old. What exact dose of an ACE inhibitor should I take right now?")
print(json.dumps(answer, indent=2, ensure_ascii=False))

{
  "status": "safety_refusal",
  "recommendation": "I’m sorry, but I can’t provide that information.",
  "supporting_evidence": [],
  "confidence": "safety_refusal",
  "missing_information": [
    "A qualified clinician must assess the individual case"
  ],
  "safety_note": "Educational information only; not a diagnosis or medical advice."
}


-----------------------------------------------------
**13. Recorded Generation Failure:**
When given an out-of-scope question, the model correctly set the status to insufficient_evidence. However, it hallucinated a claim and a fake citation in the supporting_evidence array to "prove" the topic was missing, instead of leaving the array empty.

**14. How the failure was fixed:**
I updated the GROUNDING_SYSTEM_PROMPT with a strict rule: "If the status is 'insufficient_evidence', leave the 'supporting_evidence' array completely empty [].". This successfully stopped the model from generating unnecessary citations during refusals.
